In [56]:
import pandas as pd

In [57]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [58]:
df = pd.read_csv('./datasets/zerve_events.csv')

C:\Users\anxpr\AppData\Local\Temp\ipykernel_25508\3205404977.py:1: DtypeWarning: Columns (0: person_properties.cloudProvider, 1: person_properties.purpose, 2: person_properties.role, 3: person_properties.source, 4: person_properties.work_type, 5: properties.$ai_model, 6: properties.$ai_provider, 7: properties.$ai_tools_called, 8: properties.block_type, 9: properties.block_types, 10: properties.button_name, 11: properties.connectionType, 12: properties.feature_tag, 13: properties.file_extension, 14: properties.file_type, 15: properties.link_name, 16: properties.offer_declined, 17: properties.role, 18: properties.share_platform, 19: properties.skipped, 20: properties.subscription_type, 21: properties.utm_campaign, 22: properties.utm_content, 23: properties.utm_medium, 24: properties.utm_source, 25: properties.utm_term, 26: properties.workspace_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./datasets/zerve_events.csv')


In [59]:
df.shape

(3509628, 83)

In [60]:
df.columns

Index(['person_id', 'timestamp', 'event', 'person_properties.cloudProvider',
       'person_properties.purpose', 'person_properties.role',
       'person_properties.source', 'person_properties.work_type',
       'properties.$browser', 'properties.$browser_language',
       'properties.$browser_version', 'properties.$device',
       'properties.$device_type', 'properties.$os', 'properties.$os_version',
       'properties.$geoip_continent_name', 'properties.$geoip_country_name',
       'properties.$geoip_time_zone', 'properties.$screen_height',
       'properties.$screen_width', 'properties.$ai_input_tokens',
       'properties.$ai_latency', 'properties.$ai_model',
       'properties.$ai_output_tokens', 'properties.$ai_provider',
       'properties.$ai_tool_call_count', 'properties.$ai_tools_called',
       'properties.amount', 'properties.block_type', 'properties.block_types',
       'properties.button_name', 'properties.canvas_id',
       'properties.connectionType', 'properties.credit

counting nulls under each feature

In [61]:
df_count_nulls = {}

for column in df.columns:
    df_count_nulls[column] = df[column].isnull().sum()

summary_nulls = pd.DataFrame(df_count_nulls, index=["count_nulls"]).T


In [62]:
summary_nulls.head(10)

,count_nulls
person_id,0
timestamp,0
event,0
person_properties.cloudProvider,3507489
person_properties.purpose,1497759
person_properties.role,1264978
person_properties.source,1682311
person_properties.work_type,1571686
properties.$browser,2535841
properties.$browser_language,2535841


In [63]:
print(summary_nulls["count_nulls"].unique())
print(f"number of col with nulls: {len(summary_nulls['count_nulls'].unique())}")

[      0 3507489 1497759 1264978 1682311 1571686 2535841 2538924 3422981
  549822  657429 1986064 2959810 3296653 3509554 3493063 3503970 3509075
 3306963 3509514 2575178 3507626 3508948 3507488 3047198 3509500 3509624
 3507120 3508755 3505973 3509626 3501205 3508900 3504162 3509584 3509621
 3509595 3509622 3509531 3509586 3509617 3505914 3480696 3145156 3507910
 3484576 3496749 3500737 3477472 3508461 3509623 3499692 3477576 3477639
 3069865 3470938 3473254 3454679 3474184 2775704]
number of col with nulls: 60


In [64]:
sum_null_col_count = summary_nulls.groupby("count_nulls").size().reset_index(name="num_columns").sort_values(by="num_columns", ascending=False)
sum_null_col_count[sum_null_col_count["num_columns"] > 1]

,count_nulls,num_columns
25,3477639,8
8,2535841,6
12,2959810,5
0,0,3
7,1986064,3
38,3507488,2
16,3296653,2
58,3509624,2


In [65]:
sum_null_col_count = (
    summary_nulls
    .reset_index(names="column_name")
    .groupby("count_nulls")
    .agg(
        num_columns=("column_name", "size"),
        column_names=("column_name", list)
    )
    .reset_index()
    .sort_values(by="num_columns", ascending=False)
)

sum_null_col_count[sum_null_col_count["num_columns"] > 1]


,count_nulls,num_columns,column_names
25,3477639,8,"[properties.$prev_pageview_last_content, properties.$prev_pageview_last_content_percentage, properties.$prev_pageview_last_scroll, properties.$prev_pageview_last_scroll_percentage, properties.$prev_pageview_max_content, properties.$prev_pageview_max_content_percentage, properties.$prev_pageview_max_scroll, properties.$prev_pageview_max_scroll_percentage]"
8,2535841,6,"[properties.$browser, properties.$browser_language, properties.$device_type, properties.$screen_height, properties.$screen_width, properties.$lib_rate_limit_remaining_tokens]"
12,2959810,5,"[properties.$ai_input_tokens, properties.$ai_latency, properties.$ai_model, properties.$ai_output_tokens, properties.$ai_provider]"
0,0,3,"[person_id, timestamp, event]"
7,1986064,3,"[properties.$geoip_continent_name, properties.$geoip_country_name, properties.$geoip_time_zone]"
38,3507488,2,"[properties.credits_remaining, properties.total_credits]"
16,3296653,2,"[properties.$ai_tool_call_count, properties.$ai_tools_called]"
58,3509624,2,"[properties.feature_tag, properties.subscription_type]"


## openai helper - a simple insight tool

In [66]:
import os
from pathlib import Path
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    # Fallback .env loader if python-dotenv is not installed.
    env_path = Path(".env")
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def build_datathon_context(data_dictionary_path="datasets/data_dictionary.csv", max_columns=83):
    """Build compact context about the Zerve challenge and data dictionary."""
    challenge_context = """
Zerve ODSC Datathon context:
- Challenge #1: Predict whether a user will upgrade.
- Target event: event == "subscription_upgraded".
- The model must use only information known before upgrade.
- Main risks: target leakage, using upgrade/checkout events, using post-upgrade events, or using subscription/credit fields that reveal the outcome.
- Challenge #2: Build a deterministic funnel where every user is in exactly one stage at any point in time.
- Good funnel stages must be specific, observable, complete, deterministic, and time-aware.
- Evaluation cares about feature design, leakage handling, realistic setup, transition logic, and product usefulness more than raw accuracy.
""".strip()

    path = Path(data_dictionary_path)
    if not path.exists():
        return challenge_context + "\n\nData dictionary: not found."

    data_dict = pd.read_csv(path)
    category_counts = data_dict["Category"].value_counts().to_dict()

    dictionary_rows = []
    for _, row in data_dict.head(max_columns).iterrows():
        dictionary_rows.append(
            f"- {row['Column Name']} | category={row['Category']} | null_pct={row['Null %']} | description={row['Description']}"
        )

    dictionary_context = "\n".join([
        "Data dictionary summary:",
        f"- Total documented columns: {len(data_dict)}",
        f"- Category counts: {category_counts}",
        "- Important known details:",
        "  - properties.$ai_latency is measured in seconds, not milliseconds.",
        "  - AI fields are populated only for $ai_generation events.",
        "  - properties.credits_remaining is only populated when balance reaches zero; all non-null values are 0.0.",
        "  - properties.amount is always $25.00 in this dataset.",
        "  - Credit/subscription fields are leakage-sensitive for upgrade prediction.",
        "\nColumn definitions:",
        *dictionary_rows,
    ])

    return challenge_context + "\n\n" + dictionary_context


DATATHON_CONTEXT = build_datathon_context()


def ask_openai(code_output, prompt, model="gpt-4o-mini", include_datathon_context=True):
    """
    Send a code output plus a question/prompt to an OpenAI model.

    Parameters
    ----------
    code_output : Any
        The output/value from previous analysis code, e.g. len(df["event"].unique()).
    prompt : str
        The question or instruction for the model.
    model : str
        OpenAI model name to use.
    include_datathon_context : bool
        Whether to include the Zerve challenge and data dictionary context.

    Returns
    -------
    str
        The model's text response.
    """
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("OPENAI_API_KEY was not found. Add it to your .env file first.")

    output_text = code_output if isinstance(code_output, str) else repr(code_output)

    context_text = DATATHON_CONTEXT if include_datathon_context else ""

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "system",
                "content": (
                    "You are a data science assistant helping with the Zerve ODSC datathon. "
                    "Use the provided challenge context and data dictionary. "
                    "Be concise, practical, product-oriented, and careful about target leakage."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context_text}\n\n"
                    f"Prompt:\n{prompt}\n\n"
                    f"Code output:\n```text\n{output_text}\n```"
                ),
            },
        ],
    )

    return response.output_text


Testing the openai helper

In [67]:
unique_event_count = len(df["event"].unique())

response = ask_openai(
    unique_event_count,
    "Give me the important events I need to look into for predicting subscription upgrades.",
    model="gpt-5.4-mini",
)

print(response)

For **subscription upgrade prediction**, focus on events that happen **before** the upgrade and capture meaningful intent, usage, friction, and AI engagement.

## Most important event types to inspect

### 1) Core product usage / engagement
These usually tell you whether the user is active and getting value:
- **`$pageview`**
- **workspace / canvas / app / block interaction events**  
  Look for events related to:
  - creating or editing canvases
  - adding blocks
  - running blocks / notebooks / apps
  - saving or opening workspaces
  - file upload / file access
- **`$ai_generation`**  
  Important because AI usage can strongly correlate with upgrade intent, but only use **pre-upgrade** instances.

### 2) Usage intensity / consumption
These often signal users approaching limits or high value:
- **`credits_used`**
- **events tied to credit burn or quota consumption**
- **`$exception`** or error-related events  
  Friction can predict conversion if users are blocked and then upgrade.

#

In [68]:
response = ask_openai(df["event"].unique(),
                       "give me all events in order of importance for predicting subscription upgrade",
                       model="gpt-5.4-mini",
                       )

print(response)

Here’s a practical ranking of event types for **predicting `subscription_upgraded`**, ordered by **likely predictive value** while avoiding leakage.

## 1) Highest-signal events
These usually indicate **strong intent, paywall pressure, or active product usage**.

- `credits_used`
- `credits_remaining`-related events/updates
- `subscription_checkout_started`
- `subscription_checkout_completed`
- `subscription_upgrade_clicked`
- `subscription_upgrade_viewed`
- `subscription_pricing_viewed`
- `subscription_plan_selected`
- `subscription_paywall_viewed`
- `subscription_billing_viewed`
- `trial_expired`
- `trial_end_reached`
- `payment_method_added`
- `invoice_opened`
- `workspace_limits_reached`
- `quota_exceeded`
- `usage_limit_reached`

## 2) Strong product-usage intent events
These show the user is actively using the core product and may be approaching upgrade need.

- `ai_generation`
- `ai_prompt_submitted`
- `ai_tool_used`
- `agent_worker_created`
- `agent_accept_suggestion`
- `report

In [69]:
response = ask_openai("",
                       "is there a way to get a leakage safety score for each column?",
                       model="gpt-5.4-mini",
                       )
print(response)

Yes — you can build a **column-level leakage safety score** as a practical heuristic. It won’t be a perfect proof, but it’s very useful for ranking features by risk before modeling.

## Recommended output
Score each column on something like **0–100**, where:

- **0–20 = high leakage risk**
- **21–50 = medium risk**
- **51–80 = probably safe with caution**
- **81–100 = low leakage risk**

## Simple scoring framework
For each column, start at 100 and subtract risk points based on rules:

### 1) Direct target leakage
Subtract **80–100**
- Column or value names that imply upgrade/checkout/subscription outcome
- Examples:
  - `subscription_type`
  - `amount` if it is only present on upgrade event
  - `credits_remaining` if it only appears when balance hits zero and that event is tied to upgrade behavior
  - any field populated only after the target event

### 2) Post-event or outcome-only fields
Subtract **50–90**
- Fields known to be generated only after certain events
- Example:
  - AI fi

## Analysis

Since we're focused on the events' outcome, let's start with looking into various types of events under **df['event']** that could potentially by itself or be a part of contributing factor towards subscription upgrade. We're gonna be looking into the data present under the **"event_type_upgrade_report.csv"** and the code that was used to generate this data is in the **reports / generate_event_type_report.py**.

In [ ]:
df['properties.$event_type'].unique()

<StringArray>
[nan, 'click', 'change', 'submit']
Length: 4, dtype: str

In [71]:
event_df = pd.read_csv('./reports/event_type_upgrade_report.csv')

In [73]:
event_df

,event,count,pct_rows,category,description,intuition,leakage_risk,modeling_recommendation,importance_score,importance_rank
0,promo_code_redeemed,1718,0.0490,Commercial / upgrade intent,"Commercial, billing, plan, offer, or upgrade-flow event indicating pricing intent or subscription movement.","Often directly related to conversion. Great for funnel analysis, but risky for predictive modeling unless the business question explicitly allows near-checkout intent signals.","High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.",Use for funnel diagnostics. Exclude from baseline predictive model or isolate in a separate late-intent model.,100,1
1,clicked_upgrade,1360,0.0388,Commercial / upgrade intent,"Commercial, billing, plan, offer, or upgrade-flow event indicating pricing intent or subscription movement.","Often directly related to conversion. Great for funnel analysis, but risky for predictive modeling unless the business question explicitly allows near-checkout intent signals.","High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.",Use for funnel diagnostics. Exclude from baseline predictive model or isolate in a separate late-intent model.,100,2
2,upgrade_subscription,1318,0.0376,Commercial / upgrade intent,"Commercial, billing, plan, offer, or upgrade-flow event indicating pricing intent or subscription movement.","Often directly related to conversion. Great for funnel analysis, but risky for predictive modeling unless the business question explicitly allows near-checkout intent signals.","High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.",Use for funnel diagnostics. Exclude from baseline predictive model or isolate in a separate late-intent model.,100,3
3,claim_free_offer_clicked,1269,0.0362,Commercial / upgrade intent,"Commercial, billing, plan, offer, or upgrade-flow event indicating pricing intent or subscription movement.","Often directly related to conversion. Great for funnel analysis, but risky for predictive modeling unless the business question explicitly allows near-checkout intent signals.","High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.",Use for funnel diagnostics. Exclude from baseline predictive model or isolate in a separate late-intent model.,100,4
4,agent_add_credits_button_clicked,395,0.0113,Commercial / upgrade intent,"Commercial, billing, plan, offer, or upgrade-flow event indicating pricing intent or subscription movement.","Often directly related to conversion. Great for funnel analysis, but risky for predictive modeling unless the business question explicitly allows near-checkout intent signals.","High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.",Use for funnel diagnostics. Exclude from baseline predictive model or isolate in a separate late-intent model.,100,5
5,subscription_upgraded,328,0.0093,Target outcome,The target conversion event: user upgraded to a paid subscription.,Use only to define the label and first-upgrade timestamp.,Certain leakage: this is the target and must never be used as a feature.,Label only. Exclude from features.,100,6
6,add_credits,106,0.0030,Commercial / upgrade intent,"Commercial, billing, plan, offer, or upgrade-flow event indicating pricing intent or subscription movement.","Often directly related to conversion. Great for funnel analysis, but risky for predictive modeling unless the business question explicitly allows near-checkout intent signals.","High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.",Use for funnel diagnostics. Exclude from baseline predictive model or isolate in a separate late-intent model.,100,7
7,clicked_add_credits,94,0.0027,Commercial / upgrade intent,"Commercial, billin

<StringArray>
['High leakage risk: may reveal upgrade funnel, payment intent, subscription state, or downstream commercial action.',
                                           'Certain leakage: this is the target and must never be used as a feature.',
   'Medium-high leakage risk: valuable usage signal, but may be very close to monetization or post-upgrade behavior.',
                      'Low-medium leakage if restricted to pre-upgrade observation window; strong behavioral signal.',
                                                    'Unknown/low; validate event timing and meaning before modeling.',
                                                                  'Low leakage if timestamp-filtered before upgrade.']
Length: 6, dtype: str